# XGBoost Survival AFT

Ce notebook développe, entraîne et évalue un modèle **XGBoost Survival AFT** pour prédire le temps restant avant l'insuffisance rénale terminale (IRT) chez les patients atteints de polykystose rénale autosomique dominante (PKRAD).

À partir des jeux de données dynamiques préparés précédemment, un ensemble de données est construit pour chaque nombre de visites (de 1 à 12). Les données sont ensuite préparées pour XGBoost, les hyperparamètres sont optimisés à l'aide d'Optuna, puis le modèle est entraîné et évalué sur le jeu de validation interne.

Les performances prédictives sont analysées à l'aide de plusieurs métriques, notamment le C-index, la MAE et la RMSE, afin d'étudier l'évolution de la qualité des prédictions au fur et à mesure que de nouvelles mesures longitudinales deviennent disponibles.

Enfin, une analyse d'interprétabilité basée sur les valeurs SHAP est réalisée afin d'identifier les variables les plus influentes dans les prédictions et d'améliorer la compréhension du comportement du modèle. Une validation externe est ensuite réalisée sur une cohorte indépendante afin d'évaluer la capacité de généralisation du modèle.


In [ ]:
# ==========================================================
# PACKAGES
# ==========================================================

# Standard library
import json
import os
import pickle
import warnings
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd
import polars as pl

# Model persistence
import joblib

# Hyperparameter optimization
import optuna
from optuna.samplers import TPESampler

# Machine learning
import xgboost as xgb

from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

from sklearn.inspection import permutation_importance

# Survival analysis
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

# Model interpretation
import shap

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuration
warnings.filterwarnings("ignore")

In [ ]:
!pip freeze > requirements.txt

### Importation datasets

In [ ]:
# ==================================================
# IMPORTATION DE DONNEES
# ==================================================

train_dynamic = pl.read_parquet(
    "My_data/train_dynamic.parquet"
)

internal_test_dynamic = pl.read_parquet(
    "My_data/internal_test_dynamic.parquet"
)

# La cohorte de validation externe sera utilisée
# uniquement pour la validation finale.

external_test_dynamic = pl.read_parquet(
    "My_data/external_test_dynamic.parquet"
)

In [ ]:
# ==========================================================
# CONSTRUCTION DU DATASET À k VISITES
# ==========================================================

def build_k_visit_dataset(
    df_dynamic,
    k
):

    return (

        df_dynamic

        # Conservation des snapshots disponibles jusqu'à la kᵉ visite
        .filter(
            pl.col("n_measures") <= k
        )
        .filter(
            pl.col("time_to_event") > 0
        )

        .sort([
            "patient_id",
            "n_measures"
        ])

        .group_by("patient_id")
        .last()

        .sort("patient_id")

    )


# ==========================================================
# CONSTRUCTION DES DATASETS 1 → 12 MESURES
# ==========================================================

train_datasets = {}
internal_test_datasets = {}
external_test_datasets = {}

for k in range(1, 13):

    train_datasets[k] = build_k_visit_dataset(
        train_dynamic,
        k=k
    )

    internal_test_datasets[k] = build_k_visit_dataset(
        internal_test_dynamic,
        k=k
    )

    external_test_datasets[k] = build_k_visit_dataset(
        external_test_dynamic,
        k=k
    )

    print(
        f"k = {k:2d} | "
        f"Train : {train_datasets[k].height:4d} | "
        f"Test interne : {internal_test_datasets[k].height:4d} | "
        f"Test externe : {external_test_datasets[k].height:4d}"
    )

In [ ]:
for k in range(1, 13):

    train_data = train_datasets[k]

    test_data = internal_test_datasets[k]


In [ ]:
# ==========================================================
# VARIABLES DU MODÈLE
# ==========================================================

FEATURES = [

    "sex",
    "genetic",

    "age_creat",
    "current_ckd_epi",

    "hypertension_at_t",
    "years_since_hypertension",

    "urologic_event_at_t",
    "years_since_urologic_event",

    "baseline_ckd_epi",
    "previous_ckd_epi",

    "delta_ckd_from_baseline",

    "recent_delta_ckd",
    "recent_annualized_change",

    "egfr_mean",
    "egfr_std",

    "egfr_slope",
    "recent_egfr_slope",

    "time_since_last_measure",
    "time_since_first_measure",

    "followup_density",

    "n_measures"

]

#### FONCTIONS DE LA PREPARATION DES DONNEES

In [ ]:
# ==========================================================
# PRÉPARATION DES DONNÉES
# ==========================================================

def prepare_dataset(
    train_df,
    test_df,
    features
):
    """
    Prépare les jeux d'entraînement et de test
    pour XGBoost Survival AFT.

    Parameters
    ----------
    train_df : pl.DataFrame
    test_df : pl.DataFrame
    features : list[str]

    Returns
    -------
    X_train : pandas.DataFrame
    X_test : pandas.DataFrame
    y_train : pandas.DataFrame
    y_test : pandas.DataFrame
    test_info : pandas.DataFrame
    """

    # --------------------------------------------------
    # Conversion Polars -> Pandas
    # --------------------------------------------------

    train_pd = train_df.to_pandas()

    test_pd = test_df.to_pandas()

    # --------------------------------------------------
    # Vérification des variables
    # --------------------------------------------------

    missing_train = (
        set(features)
        - set(train_pd.columns)
    )

    missing_test = (
        set(features)
        - set(test_pd.columns)
    )

    assert len(missing_train) == 0, (
        f"Variables manquantes dans train : {missing_train}"
    )

    assert len(missing_test) == 0, (
        f"Variables manquantes dans test : {missing_test}"
    )

    # --------------------------------------------------
    # Colonnes de survie
    # --------------------------------------------------

    target_columns = [

        "patient_id",

        "time_to_event",

        "event_irt"

    ]

    # --------------------------------------------------
    # Variables explicatives
    # --------------------------------------------------

    X_train = train_pd[
        features
    ].copy()

    X_test = test_pd[
        features
    ].copy()

    categorical_features = [
        "sex",
        "genetic",
        "hypertension_at_t",
        "urologic_event_at_t",
    ]

    for col in categorical_features:

        X_train[col] = X_train[col].astype("category")
        X_test[col] = X_test[col].astype("category")

    # --------------------------------------------------
    # Variables de survie
    # --------------------------------------------------

    y_train = train_pd[
        target_columns
    ].copy()

    y_test = test_pd[
        target_columns
    ].copy()

    # --------------------------------------------------
    # Informations nécessaires à l'évaluation
    # --------------------------------------------------

    test_info = test_pd[

        target_columns
        + ["n_measures"]

    ].copy()

    return (

        X_train,

        X_test,

        y_train,

        y_test,

        test_info

    )

In [ ]:
# ==========================================================
# LABELS POUR XGBOOST SURVIVAL AFT
# ==========================================================

def prepare_aft_labels(y):

    """
    Construit les bornes inférieure et supérieure
    requises par XGBoost Survival AFT.
    """

    # --------------------------------------------------
    # Vérification des données
    # --------------------------------------------------

    assert set(y["event_irt"].unique()) <= {0, 1}, (
        "event_irt doit contenir uniquement 0 (censuré) et 1 (événement)."
    )

    assert (y["time_to_event"] > 0).all(), (
        "time_to_event doit être strictement positif."
    )

    # --------------------------------------------------
    # Temps observé
    # --------------------------------------------------

    lower_bound = (
        y["time_to_event"]
        .to_numpy(dtype=float)
    )

    upper_bound = (
        lower_bound.copy()
    )

    # --------------------------------------------------
    # Patients censurés
    # --------------------------------------------------

    censored = (
        y["event_irt"] == 0
    )

    upper_bound[censored] = np.inf

    return (
        lower_bound,
        upper_bound
    )

In [ ]:
# ==========================================================
# CONSTRUCTION D'UNE DMATRIX XGBOOST SURVIVAL AFT
# ==========================================================

def build_dmatrix(
    X,
    lower_bound,
    upper_bound
):
    """
    Construit une DMatrix XGBoost contenant les bornes
    inférieure et supérieure requises par le modèle
    Survival AFT.

    Paramètres
    ----------
    X : pandas.DataFrame ou np.ndarray
        Variables explicatives.

    lower_bound : np.ndarray
        Temps observé (borne inférieure).

    upper_bound : np.ndarray
        Temps observé ou +∞ pour les patients censurés.

    Returns
    -------
    xgb.DMatrix
    """

    # --------------------------------------------------
    # Vérifications
    # --------------------------------------------------

    assert (
        len(X)
        == len(lower_bound)
        == len(upper_bound)
    ), "Dimensions incompatibles."

    assert np.all(
        lower_bound <= upper_bound
    ), "Certaines bornes inférieures sont supérieures aux bornes supérieures."

    # --------------------------------------------------
    # Construction de la DMatrix
    # --------------------------------------------------

    dmatrix = xgb.DMatrix(
        X,
        enable_categorical=True
    )

    # --------------------------------------------------
    # Labels Survival AFT
    # --------------------------------------------------

    dmatrix.set_float_info(
        "label_lower_bound",
        lower_bound
    )

    dmatrix.set_float_info(
        "label_upper_bound",
        upper_bound
    )

    return dmatrix

### FONCTION D'EVALUATION

In [ ]:
# ==========================================================
# ÉVALUATION DES PRÉDICTIONS
# ==========================================================

def evaluate_predictions(

    train_info,

    test_info,

    pred_time,

    k

):

    # ------------------------------------------------------
    # DataFrame des prédictions
    # ------------------------------------------------------

    df_pred = pd.DataFrame({

        "patient_id":

            test_info["patient_id"],

        "n_measures":

            test_info["n_measures"],

        "time_observed":

            test_info["time_to_event"],

        "event_irt":

            test_info["event_irt"],

        "pred_time":

            pred_time

    })

    # ------------------------------------------------------
    # Patients ayant réellement atteint k visites
    # ------------------------------------------------------

    df_pred = df_pred.loc[

        df_pred["n_measures"] >= k

    ].copy()

    # ------------------------------------------------------
    # Patients avec une prédiction valide
    # ------------------------------------------------------

    df_valid = df_pred.dropna(

        subset=["pred_time"]

    ).copy()

    # ------------------------------------------------------
    # C-index d'Uno
    # ------------------------------------------------------

    try:

        # Données de survie du jeu d'entraînement
        y_train_surv = Surv.from_arrays(

            event=train_info["event_irt"].astype(bool),

            time=train_info["time_to_event"]

        )

        # Données de survie du jeu évalué
        y_test_surv = Surv.from_arrays(

            event=df_valid["event_irt"].astype(bool),

            time=df_valid["time_observed"]

        )

        cindex = concordance_index_ipcw(

            y_train_surv,

            y_test_surv,

            -df_valid["pred_time"]

        )[0]

    except ValueError:

        cindex = np.nan

    # ------------------------------------------------------
    # Patients avec événement
    # ------------------------------------------------------

    df_event = df_valid.loc[

        df_valid["event_irt"] == 1

    ].copy()

    # ------------------------------------------------------
    # MAE / RMSE
    # ------------------------------------------------------

    if len(df_event) > 0:

        mae = mean_absolute_error(

            df_event["time_observed"],

            df_event["pred_time"]

        )

        rmse = np.sqrt(

            mean_squared_error(

                df_event["time_observed"],

                df_event["pred_time"]

            )

        )

    else:

        mae = np.nan

        rmse = np.nan

    # ------------------------------------------------------
    # Résultats
    # ------------------------------------------------------

    return {

        "predictions":

            df_event,

        "cindex":

            cindex,

        "mae":

            mae,

        "rmse":

            rmse,

        "n_patients":

            len(df_pred),

        "n_events":

            int(

                df_pred["event_irt"].sum()

            ),

        "n_predicted":

            len(df_event)

    }

#### PARAMETRES DU MODELE

In [ ]:
# ==========================================================
# XGBOOST SURVIVAL AFT
# ==========================================================

SEED = 42
FIXED_XGB_PARAMS = {

    "objective": "survival:aft",

    "eval_metric": "aft-nloglik",

    "tree_method": "hist",

    "seed": SEED

}


# ==========================================================
# PARAMÈTRES DE DÉPART
# ==========================================================

INITIAL_XGB_PARAMS = {

    **FIXED_XGB_PARAMS,

    "aft_loss_distribution": "normal",

    "aft_loss_distribution_scale": 0.25,

    "learning_rate": 0.03,

    "max_depth": 3,

    "min_child_weight": 1,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "lambda": 1.0,

    "alpha": 0.0

}

#### CALIBRATION DES HYPERPARAMETRES

In [ ]:
# ==========================================================
# VALIDATION CROISÉE
# ==========================================================

def evaluate_cv(

    dataset,

    features,

    xgb_params,

    metric="cindex",

    n_splits=5,

    num_boost_round=200

):

    # ------------------------------------------------------
    # Préparation des données
    # ------------------------------------------------------

    (
        X,
        _,
        y,
        _,
        _
    ) = prepare_dataset(

        train_df=dataset,

        test_df=dataset,

        features=features

    )

    # ------------------------------------------------------
    # Validation croisée
    # ------------------------------------------------------

    cv = StratifiedKFold(

        n_splits=n_splits,

        shuffle=True,

        random_state=SEED

    )

    scores = []

    # ------------------------------------------------------
    # Boucle sur les folds
    # ------------------------------------------------------

    for train_idx, valid_idx in cv.split(

        X,

        y["event_irt"]

    ):

        # --------------------------------------------------
        # Split
        # --------------------------------------------------

        X_train = X.iloc[train_idx].copy()

        X_valid = X.iloc[valid_idx].copy()

        y_train = y.iloc[train_idx].copy()

        y_valid = y.iloc[valid_idx].copy()

        # --------------------------------------------------
        # Labels AFT
        # --------------------------------------------------

        train_lower, train_upper = prepare_aft_labels(
            y_train
        )

        valid_lower, valid_upper = prepare_aft_labels(
            y_valid
        )

        # --------------------------------------------------
        # DMatrix
        # --------------------------------------------------

        dtrain = build_dmatrix(

            X_train,

            train_lower,

            train_upper

        )

        dvalid = build_dmatrix(

            X_valid,

            valid_lower,

            valid_upper

        )

        # --------------------------------------------------
        # Entraînement
        # --------------------------------------------------

        booster = xgb.train(

            params=xgb_params,

            dtrain=dtrain,

            num_boost_round=num_boost_round,

            verbose_eval=False

        )

        # --------------------------------------------------
        # Prédictions
        # --------------------------------------------------

        pred = booster.predict(
            dvalid
        )

        # --------------------------------------------------
        # Vérification des prédictions
        # --------------------------------------------------

        if not np.all(np.isfinite(pred)):

            raise optuna.TrialPruned()

        pred = np.asarray(
            pred,
            dtype=np.float64
        )

        # --------------------------------------------------
        # C-index d'Uno
        # --------------------------------------------------

        if metric == "cindex":

            # Données de survie du fold d'entraînement
            y_train_surv = Surv.from_arrays(

                event=y_train[
                    "event_irt"
                ].astype(bool),

                time=y_train[
                    "time_to_event"
                ]

            )

            # Données de survie du fold de validation
            y_valid_surv = Surv.from_arrays(

                event=y_valid[
                    "event_irt"
                ].astype(bool),

                time=y_valid[
                    "time_to_event"
                ]

            )

            try:

                score = concordance_index_ipcw(

                    y_train_surv,

                    y_valid_surv,

                    -pred

                )[0]

            except ValueError:

                score = np.nan

        # --------------------------------------------------
        # MAE
        # --------------------------------------------------

        elif metric == "mae":

            event_mask = (

                y_valid[
                    "event_irt"
                ] == 1

            )

            if event_mask.sum() == 0:

                continue

            score = mean_absolute_error(

                y_valid.loc[
                    event_mask,
                    "time_to_event"
                ],

                pred[event_mask]

            )

        else:

            raise ValueError(

                "metric doit être 'cindex' ou 'mae'."

            )

        scores.append(score)

    # ------------------------------------------------------
    # Score moyen
    # ------------------------------------------------------

    if metric == "cindex":

        return float(
            np.nanmean(scores)
        )

    else:

        return float(
            np.mean(scores)
        )

In [ ]:
# ==========================================================
# FONCTION OBJECTIF OPTUNA
# ==========================================================

def objective(

    trial,

    dataset,

    features,

    metric="cindex"

):

    # ------------------------------------------------------
    # Hyperparamètres à optimiser
    # ------------------------------------------------------

    params = {

        **FIXED_XGB_PARAMS,

        "learning_rate": trial.suggest_float(

            "learning_rate",

            0.01,

            0.08,

            log=True

        ),

        "max_depth": trial.suggest_int(

            "max_depth",

            2,

            6

        ),

        "min_child_weight": trial.suggest_int(

            "min_child_weight",

            1,

            10

        ),

        "subsample": trial.suggest_float(

            "subsample",

            0.5,

            1.0

        ),

        "colsample_bytree": trial.suggest_float(

            "colsample_bytree",

            0.5,

            1.0

        ),

        "lambda": trial.suggest_float(

            "lambda",

            1e-3,

            10,

            log=True

        ),

        "alpha": trial.suggest_float(

            "alpha",

            1e-3,

            10,

            log=True

        ),

        "aft_loss_distribution": trial.suggest_categorical(

            "aft_loss_distribution",

            [

                "normal",

                "logistic",

                "extreme"

            ]

        ),

        "aft_loss_distribution_scale": trial.suggest_float(

            "aft_loss_distribution_scale",

            0.1,

            1.0,

            log=True

        )

    }

    # ------------------------------------------------------
    # Validation croisée
    # ------------------------------------------------------

    score = evaluate_cv(

        dataset=dataset,

        features=features,

        xgb_params=params,

        metric=metric

    )

    # ------------------------------------------------------
    # Score Optuna
    # ------------------------------------------------------

    if metric == "cindex":

        return score

    else:

        return score

In [ ]:
# ==========================================================
# OPTIMISATION DES HYPERPARAMÈTRES
# ==========================================================

def optimize_xgb_hyperparameters(

    dataset,

    features,

    metric="cindex",

    n_trials=50,

    timeout=None

):

    # ------------------------------------------------------
    # Sens de l'optimisation
    # ------------------------------------------------------

    direction = (

        "maximize"

        if metric == "cindex"

        else "minimize"

    )

    # ------------------------------------------------------
    # Étude Optuna
    # ------------------------------------------------------

    study = optuna.create_study(

        direction=direction,

        sampler=TPESampler(

            seed=SEED

        )

    )

    # ------------------------------------------------------
    # Optimisation
    # ------------------------------------------------------

    study.optimize(

        lambda trial: objective(

            trial=trial,

            dataset=dataset,

            features=features,

            metric=metric

        ),

        n_trials=n_trials,

        timeout=timeout,

        show_progress_bar=True

    )

    # ------------------------------------------------------
    # Meilleurs paramètres
    # ------------------------------------------------------

    best_params = {

        **FIXED_XGB_PARAMS,

        **study.best_params

    }

    # ------------------------------------------------------
    # Résultats
    # ------------------------------------------------------

    print("\n==============================")

    print("OPTUNA TERMINÉ")

    print("==============================")

    print(f"Métrique : {metric}")

    print(f"Meilleur score : {study.best_value:.5f}")

    print("\nMeilleurs paramètres :\n")

    for key, value in best_params.items():

        print(f"{key:<30} : {value}")

    # ------------------------------------------------------
    # Sauvegarde
    # ------------------------------------------------------

    os.makedirs(

        "params",

        exist_ok=True

    )

    with open(

        f"params/best_xgb_params_{metric}.json",

        "w",

        encoding="utf-8"

    ) as f:

        json.dump(

            best_params,

            f,

            indent=4

        )

    joblib.dump(

        study,

        f"params/optuna_study_{metric}.pkl"

    )

    # ------------------------------------------------------
    # Retour
    # ------------------------------------------------------

    return (

        best_params,

        study

    )

In [ ]:
# ======================================
# OPTIMISATION 
# ======================================

BEST_XGB_PARAMS, study = optimize_xgb_hyperparameters(

    dataset=train_datasets[12],

    features=FEATURES,

    metric="mae",

    n_trials=50

)

In [ ]:
## sauvegarde les hyperparamètres


# ==========================================================
# Dossier de sauvegarde
# ==========================================================

output_dir = (
    r".\models\xgboost"
)

os.makedirs(output_dir, exist_ok=True)

# ==========================================================
# Sauvegarde des meilleurs hyperparamètres
# ==========================================================

joblib.dump(
    BEST_XGB_PARAMS,
    os.path.join(output_dir, "best_xgb_params.pkl")
)

# ==========================================================
# Sauvegarde de l'étude Optuna
# ==========================================================

joblib.dump(
    study,
    os.path.join(output_dir, "xgb_optuna_study.pkl")
)

print("Hyperparamètres et étude Optuna sauvegardés.")

In [ ]:
# recharge des hyperparamètres sans tourner l'optuna


output_dir = (
    r".\models\xgboost"
)

BEST_XGB_PARAMS = joblib.load(
    os.path.join(output_dir, "best_xgb_params.pkl")
)

study = joblib.load(
    os.path.join(output_dir, "xgb_optuna_study.pkl")
)

print(BEST_XGB_PARAMS)
print("Nombre de trials :", len(study.trials))

#### FONCTION D'ENTRAINEMENT

In [ ]:
# ==========================================================
# XGBOOST SURVIVAL AFT
# ==========================================================

def run_xgb_survival(

    train_dataset,

    test_dataset,

    features,

    k,

    xgb_params=None,

    num_boost_round=200,

    model_name="LONGITUDINAL",

    verbose=True

):

    # ------------------------------------------------------
    # Paramètres
    # ------------------------------------------------------

    if xgb_params is None:

        xgb_params = BEST_XGB_PARAMS

    # ------------------------------------------------------
    # Préparation des données
    # ------------------------------------------------------

    (
        X_train,
        X_test,
        y_train_df,
        y_test_df,
        test_info

    ) = prepare_dataset(

        train_dataset,

        test_dataset,

        features

    )

    # ------------------------------------------------------
    # Vérification des dimensions
    # ------------------------------------------------------

    assert len(X_train) == len(y_train_df)

    assert len(X_test) == len(y_test_df)

    # ------------------------------------------------------
    # Labels AFT
    # ------------------------------------------------------

    train_lower, train_upper = prepare_aft_labels(
        y_train_df
    )

    test_lower, test_upper = prepare_aft_labels(
        y_test_df
    )

    # ------------------------------------------------------
    # DMatrix
    # ------------------------------------------------------

    dtrain = build_dmatrix(

        X_train,

        train_lower,

        train_upper

    )

    dtest = build_dmatrix(

        X_test,

        test_lower,

        test_upper

    )

    # ------------------------------------------------------
    # Entraînement
    # ------------------------------------------------------

    booster = xgb.train(

        params=xgb_params,

        dtrain=dtrain,

        num_boost_round=num_boost_round,

        evals=[

            (dtrain, "train"),

            (dtest, "test")

        ],

        verbose_eval=False

    )

    # ------------------------------------------------------
    # Prédictions
    # ------------------------------------------------------

    pred_train = booster.predict(
        dtrain
    )

    pred_test = booster.predict(
        dtest
    )

    # ------------------------------------------------------
# Informations train pour le C-index d'Uno
# ------------------------------------------------------

train_info = y_train_df[
    [
        "time_to_event",
        "event_irt"
    ]
].copy()

    # ------------------------------------------------------
    # C-index train - Uno
    # ------------------------------------------------------

    try:

        y_train_surv = Surv.from_arrays(

            event=y_train_df[
                "event_irt"
            ].astype(bool),

            time=y_train_df[
                "time_to_event"
            ]

        )

        cindex_train = concordance_index_ipcw(

            y_train_surv,

            y_train_surv,

            -pred_train

        )[0]

    except ValueError:

        cindex_train = np.nan

    # ------------------------------------------------------
    # Évaluation
    # ------------------------------------------------------

    evaluation = evaluate_predictions(

        train_info=y_train_df,

        test_info=test_info,

        pred_time=pred_test,

        k=k

    )

    # ------------------------------------------------------
    # Historique des prédictions
    # ------------------------------------------------------

    history = (

        evaluation["predictions"]

        .copy()

        .assign(

            n_visits=k

        )

    )

    # ------------------------------------------------------
    # Affichage
    # ------------------------------------------------------

    if verbose:

        print("=" * 60)

        print(
            f"XGB Survival AFT - {model_name}"
        )

        print("=" * 60)

        print(
            f"C-index train : {cindex_train:.4f}"
        )

        print(
            f"C-index test  : "
            f"{evaluation['cindex']:.4f}"
        )

        print(
            f"Patients test : "
            f"{evaluation['n_patients']}"
        )

        print(
            f"Événements IRT : "
            f"{evaluation['n_events']}"
        )

        print(
            f"Patients évalués : "
            f"{evaluation['n_predicted']}"
        )

        print(
            f"MAE : "
            f"{evaluation['mae']:.2f} années"
        )

        print(
            f"RMSE : "
            f"{evaluation['rmse']:.2f} années"
        )

    # ------------------------------------------------------
    # Résultats
    # ------------------------------------------------------

    return {

        "model":

            booster,

        "X_train":

            X_train,

        "X_test":

            X_test,

        "test_info":

            test_info,

        "history":

            history,

        "event_predictions":

            evaluation["predictions"],

        "features":

            features,

        "xgb_params":

            xgb_params,

        "num_boost_round":

            num_boost_round,

        "cindex_train":

            cindex_train,

        "cindex_test":

            evaluation["cindex"],

        "mae":

            evaluation["mae"],

        "rmse":

            evaluation["rmse"],

        "n_patients":

            evaluation["n_patients"],

        "n_events":

            evaluation["n_events"],

        "n_predicted":

            evaluation["n_predicted"]

    }

In [ ]:
# ==========================================================
# ÉVALUATION DES MODÈLES XGBOOST SURVIVAL AFT
# ==========================================================

results_xgb_internal = []

predictions_xgb = {}

models_xgb = {}

X_train_xgb = {}

X_test_xgb = {}

test_info_xgb = {}

for k in range(1, 13):

    res = run_xgb_survival(

        train_dataset=train_datasets[k],

        test_dataset=internal_test_datasets[k],

        features=FEATURES,

        k=k,

        verbose=False

    )

    models_xgb[k] = res["model"]

    X_train_xgb[k] = res["X_train"]

    X_test_xgb[k] = res["X_test"]

    test_info_xgb[k] = res["test_info"]

    predictions_xgb[k] = res["event_predictions"]

    results_xgb_internal.append({

        "model": "XGB",

        "n_visits": k,

        "n_patients": res["n_patients"],

        "n_events": res["n_events"],

        "n_predicted": res["n_predicted"],

        "cindex_train": res["cindex_train"],

        "cindex_test": res["cindex_test"],

        "mae": res["mae"],

        "rmse": res["rmse"]

    })

results_xgb_internal = pd.DataFrame(
    results_xgb_internal
)

### Évaluation globale des performances

In [ ]:
display(
    results_xgb_internal.round(3)
)

In [ ]:
# ===================================================
# Évolution du C-index
# ===================================================


# ==========================================================
# Libellés de l'axe des x
# ==========================================================
tickvals = results_xgb["n_visits"]

ticktext = [
    f"{visit} (n={patient})"
    for visit, patient in zip(
        results_xgb["n_visits"],
        results_xgb["n_patients"]
    )
]


# ==========================================================
# Figure
# ==========================================================
fig = px.line(
    results_xgb,
    x="n_visits",
    y="cindex_test",
    markers=True,
    template="simple_white"
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=9)
)

fig.update_xaxes(
    tickmode="array",
    tickvals=tickvals,
    ticktext=ticktext,
    title="Nombre de visites (nombre de patients)"
)

fig.update_layout(

    title="Évolution du C-index selon le nombre de visites",

    yaxis_title="C-index",

    yaxis=dict(
        range=[0.75, 0.90]
    ),

    width=850,
    height=450,

    font=dict(size=16),

    margin=dict(
        t=70,
        b=80,
        l=70,
        r=30
    )
)

fig.show()

In [ ]:
# ==========================================================
# Libellés de l'axe des x
# ==========================================================
tickvals = results_xgb["n_visits"]

ticktext = [
    f"{visit} (n={patient})"
    for visit, patient in zip(
        results_xgb["n_visits"],
        results_xgb["n_patients"]
    )
]

# ==========================================================
# Figure
# ==========================================================
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=results_xgb["n_visits"],
        y=results_xgb["mae"],
        mode="lines+markers",
        name="MAE",
        line=dict(width=3),
        marker=dict(size=8)
    )
)

fig.add_trace(
    go.Scatter(
        x=results_xgb["n_visits"],
        y=results_xgb["rmse"],
        mode="lines+markers",
        name="RMSE",
        line=dict(width=3),
        marker=dict(size=8)
    )
)

fig.update_xaxes(
    tickmode="array",
    tickvals=tickvals,
    ticktext=ticktext,
    title="Nombre de visites"
)

fig.update_layout(

    template="simple_white",

    title="Évolution des erreurs de prédiction selon le nombre de visites",

    yaxis_title="Erreur (années)",

    width=850,
    height=450,

    font=dict(size=16),

    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.08
    ),

    margin=dict(
        t=70,
        b=80,
        l=70,
        r=30
    )
)

fig.show()

In [ ]:
# ==========================================================
# CONSTRUCTION DE L'HISTORIQUE DES PRÉDICTIONS
# ==========================================================

history = []

for k in range(1, 13):

    df = predictions_xgb[k].copy()

    df["n_visits"] = k

    history.append(df)

history = pd.concat(history, ignore_index=True)

# ==========================================================
# ERREURS DE PRÉDICTION
# ==========================================================

# Erreur signée : positive = surestimation, négative = sous-estimation
history["error"] = history["pred_time"] - history["time_observed"]

# Erreur absolue
history["abs_error"] = history["error"].abs()

history.head()

In [ ]:
# Patients présents au snapshot 12
patients_12 = history.loc[history["n_visits"] == 12, "patient_id"].unique()

print(f"{len(patients_12)} patients ont 12 visites.")

# Choix du premier patient
patient_id = patients_12[0]

print("Patient choisi :", patient_id)

### ANALYSE DES PRÉDICTIONS

In [ ]:
patient_history = (
    history
    .loc[history["patient_id"] == patient_id]
    .sort_values("n_visits")
)

patient_history

In [ ]:
# ==========================================================
# FONCTION PRINCIPALE
# ==========================================================

def plot_snapshot(snapshot):

    # ------------------------------------------------------
    # Patients présents au snapshot
    # ------------------------------------------------------

    patients = predictions_xgb[snapshot]["patient_id"].unique()

    df = (
        history[
            history["patient_id"].isin(patients)
        ]
        .query("n_visits <= @snapshot")
        .sort_values(
            ["patient_id", "n_visits"]
        )
        .copy()
    )

    # ------------------------------------------------------
    # Résumé statistique
    # ------------------------------------------------------

    summary = (
        df
        .groupby("n_visits")
        .agg(
            n_patients=("patient_id", "nunique"),
            n_events=("event_irt", "sum"),
            observed_mean=("time_observed", "mean"),
            predicted_mean=("pred_time", "mean"),
            error_mean=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("error", lambda x: np.sqrt(np.mean(x ** 2)))
        )
        .reset_index()
    )

    summary["n_censored"] = (
        summary["n_patients"]
        - summary["n_events"]
    )

    # ------------------------------------------------------
    # Figure
    # ------------------------------------------------------

    fig = go.Figure()

    # ======================================================
    # COURBES OBSERVÉES INDIVIDUELLES
    # ======================================================

    for _, patient in df.groupby("patient_id"):

        fig.add_trace(

            go.Scatter(

                x=patient["n_visits"],

                y=patient["time_observed"],

                mode="lines",

                line=dict(
                    color="rgba(0,0,0,0.08)",
                    width=1
                ),

                hoverinfo="skip",

                showlegend=False

            )

        )

    # ======================================================
    # COURBES PRÉDITES INDIVIDUELLES
    # ======================================================

    for _, patient in df.groupby("patient_id"):

        fig.add_trace(

            go.Scatter(

                x=patient["n_visits"],

                y=patient["pred_time"],

                mode="lines",

                line=dict(
                    color="rgba(220,20,60,0.08)",
                    width=1
                ),

                hoverinfo="skip",

                showlegend=False

            )

        )

    # ======================================================
    # COURBES MOYENNES
    # ======================================================

    custom = np.stack([
        summary["n_patients"],
        summary["n_events"],
        summary["n_censored"],
        summary["observed_mean"],
        summary["predicted_mean"],
        summary["error_mean"],
        summary["mae"],
        summary["rmse"]
    ], axis=1)

    fig.add_trace(

        go.Scatter(

            x=summary["n_visits"],

            y=summary["observed_mean"],

            mode="lines+markers",

            name="Temps observé moyen",

            line=dict(
                color="black",
                width=4
            ),

            marker=dict(size=9),

            customdata=custom,

            hovertemplate=
            "<b>Temps observé moyen</b><br><br>"
            "Visite : %{x}<br>"
            "Patients : %{customdata[0]:.0f}<br>"
            "Evénements : %{customdata[1]:.0f}<br>"
            "Censurés : %{customdata[2]:.0f}<br><br>"
            "Temps observé : %{customdata[3]:.2f} ans<br>"
            "Temps prédit : %{customdata[4]:.2f} ans<br>"
            "Erreur moyenne : %{customdata[5]:+.2f} ans<br>"
            "MAE : %{customdata[6]:.2f} ans<br>"
            "RMSE : %{customdata[7]:.2f} ans"
            "<extra></extra>"

        )

    )

    fig.add_trace(

        go.Scatter(

            x=summary["n_visits"],

            y=summary["predicted_mean"],

            mode="lines+markers",

            name="Temps prédit moyen",

            line=dict(
                color="firebrick",
                width=4
            ),

            marker=dict(size=9),

            customdata=custom,

            hovertemplate=
            "<b>Temps prédit moyen</b><br><br>"
            "Visite : %{x}<br>"
            "Patients : %{customdata[0]:.0f}<br>"
            "Evénements : %{customdata[1]:.0f}<br>"
            "Censurés : %{customdata[2]:.0f}<br><br>"
            "Temps observé : %{customdata[3]:.2f} ans<br>"
            "Temps prédit : %{customdata[4]:.2f} ans<br>"
            "Erreur moyenne : %{customdata[5]:+.2f} ans<br>"
            "MAE : %{customdata[6]:.2f} ans<br>"
            "RMSE : %{customdata[7]:.2f} ans"
            "<extra></extra>"

        )

    )

    # ======================================================
    # MISE EN FORME
    # ======================================================

    fig.update_layout(

        template="plotly_white",

        width=1000,

        height=700,

        title=f"Snapshot {snapshot}",

        xaxis_title="Nombre de visites",

        yaxis_title="Temps restant avant IRT (années)",

        hovermode="closest",

        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center",
            y=1.03
        )

    )

    fig.update_xaxes(

        dtick=1,

        range=[0.8, snapshot + 0.2]

    )

    fig.show()


# ==========================================================
# WIDGET
# ==========================================================

dropdown = widgets.Dropdown(

    options=list(range(2, 13)),

    value=2,

    description="Snapshot :",

    style={"description_width": "initial"},

    layout=widgets.Layout(width="250px")

)

# Zone d'affichage de la figure
output = widgets.Output()


def update(change=None):

    with output:

        output.clear_output(wait=True)

        plot_snapshot(dropdown.value)


dropdown.observe(update, names="value")

display(dropdown)
display(output)

update()

In [ ]:
# =====================================================
# Widget Output
# =====================================================

out = widgets.Output()

# =====================================================
# Fonction
# =====================================================

def plot_error_distribution(snapshot):

    with out:

        clear_output(wait=True)

        df = (
            history
            .query("n_visits == @snapshot")
            .copy()
        )

        mae = df["abs_error"].mean()
        rmse = (df["error"] ** 2).mean() ** 0.5
        bias = df["error"].mean()

        fig = px.histogram(

            df,

            x="error",

            nbins=40,

            marginal="box",

            opacity=0.8,

            title=f"Distribution des erreurs - Snapshot {snapshot}"

        )

        fig.add_vline(

            x=0,

            line_dash="dash",

            line_color="red",

            annotation_text="Erreur nulle"

        )

        fig.update_layout(

            template="plotly_white",

            width=1000,

            height=600,

            bargap=0.05,

            xaxis_title="Erreur (temps prédit - temps observé) [années]",

            yaxis_title="Nombre de patients",

            annotations=[

                dict(

                    x=0.99,

                    y=0.98,

                    xref="paper",

                    yref="paper",

                    xanchor="right",

                    yanchor="top",

                    showarrow=False,

                    bgcolor="white",

                    bordercolor="black",

                    borderwidth=1,

                    text=(
                        f"<b>Patients :</b> {len(df)}<br>"
                        f"<b>Biais :</b> {bias:.2f} ans<br>"
                        f"<b>MAE :</b> {mae:.2f} ans<br>"
                        f"<b>RMSE :</b> {rmse:.2f} ans"
                    )

                )

            ]

        )

        fig.show()


# =====================================================
# Dropdown
# =====================================================

dropdown = widgets.Dropdown(

    options=[(f"Snapshot {i}", i) for i in range(1, 13)],

    value=1,

    description="Snapshot :",

    style={"description_width": "initial"},

    layout=widgets.Layout(width="250px")

)

# =====================================================
# Callback
# =====================================================

def update(change):

    if change["name"] == "value":

        plot_error_distribution(change["new"])


dropdown.observe(update, names="value")

# =====================================================
# Affichage
# =====================================================

display(dropdown)
display(out)

plot_error_distribution(dropdown.value)

In [ ]:
fig = px.box(

    history,

    x="n_visits",

    y="error",

    points="outliers",

    title="Distribution des erreurs selon le nombre de visites"

)

fig.add_hline(

    y=0,

    line_dash="dash",

    line_color="red"

)

fig.update_layout(

    template="plotly_white",

    xaxis_title="Nombre de visites",

    yaxis_title="Erreur (années)"

)

fig.show()

In [ ]:
fig = px.box(

    history,

    x="n_visits",

    y="abs_error",

    points="outliers",

    title="Erreur absolue selon le nombre de visites"

)

fig.update_layout(

    template="plotly_white",

    xaxis_title="Nombre de visites",

    yaxis_title="Erreur absolue (années)"

)

fig.show()

In [ ]:
out = widgets.Output()


def plot_calibration(snapshot):

    with out:

        clear_output(wait=True)

        df = (
            history
            .query("n_visits == @snapshot")
            .copy()
        )

        mae = df["abs_error"].mean()
        rmse = np.sqrt((df["error"]**2).mean())
        bias = df["error"].mean()

        fig = px.scatter(

            df,

            x="time_observed",

            y="pred_time",

            opacity=0.7,

            trendline="ols",

            labels={

                "time_observed": "Temps observé avant IRT (années)",

                "pred_time": "Temps prédit (années)"

            },

            title=f"Calibration - Snapshot {snapshot}"

        )

        # diagonale parfaite

        mini = min(df["time_observed"].min(),
                   df["pred_time"].min())

        maxi = max(df["time_observed"].max(),
                   df["pred_time"].max())

        fig.add_trace(

            go.Scatter(

                x=[mini, maxi],

                y=[mini, maxi],

                mode="lines",

                line=dict(
                    color="red",
                    dash="dash"
                ),

                name="Prédiction parfaite"

            )

        )

        fig.update_layout(

            template="plotly_white",

            width=900,

            height=700,

            annotations=[

                dict(

                    x=0.98,

                    y=0.02,

                    xref="paper",

                    yref="paper",

                    xanchor="right",

                    showarrow=False,

                    bgcolor="white",

                    bordercolor="black",

                    text=(
                        f"Patients : {len(df)}<br>"
                        f"Biais : {bias:.2f}<br>"
                        f"MAE : {mae:.2f}<br>"
                        f"RMSE : {rmse:.2f}"
                    )

                )

            ]

        )

        fig.show()


dropdown = widgets.Dropdown(

    options=[(f"Snapshot {i}", i) for i in range(1,13)],

    value=1,

    description="Snapshot"

)

dropdown.observe(
    lambda change: plot_calibration(change["new"]),
    names="value"
)

display(dropdown)
display(out)

plot_calibration(1)

### Analyse des prédictions selon les variables

In [ ]:
def subgroup_performance(
    train_df,
    test_df,
    group_col,
    prediction_col="pred_time",
    target_col="time_observed",
    event_col="event_irt",
    events_only=True,
):
    """
    Résumé des performances par sous-groupe.

    Parameters
    ----------
    train_df : pd.DataFrame
    test_df : pd.DataFrame
    group_col : str
        Variable définissant les sous-groupes.
    prediction_col : str, default="pred_time"
        Colonne contenant les prédictions.
    target_col : str, default="time_observed"
        Colonne contenant le temps observé.
    event_col : str, default="event_irt"
        Variable d'événement (0=censuré, 1=IRT).
    events_only : bool, default=True
        Si True, les métriques sont calculées uniquement sur les événements.

    Returns
    -------
    pd.DataFrame
    """

    groups = (
        pd.concat([train_df[group_col], test_df[group_col]])
        .dropna()
        .unique()
    )

    rows = []

    for g in groups:

        train_g = train_df.loc[train_df[group_col] == g]
        test_g = test_df.loc[test_df[group_col] == g]

        # ==========================
        # Comptages
        # ==========================
        n_train = len(train_g)
        n_event_train = int(train_g[event_col].sum())

        n_test = len(test_g)
        n_event_test = int(test_g[event_col].sum())

        event_rate_train = (
            100 * n_event_train / n_train
            if n_train > 0 else np.nan
        )

        event_rate_test = (
            100 * n_event_test / n_test
            if n_test > 0 else np.nan
        )

        # ==========================
        # Patients utilisés pour l'évaluation
        # ==========================
        if events_only:
            test_eval = test_g.loc[test_g[event_col] == 1]
        else:
            test_eval = test_g

        n_eval = len(test_eval)

        # ==========================
        # Métriques
        # ==========================
        if n_eval > 0:

            error = (
                test_eval[prediction_col]
                - test_eval[target_col]
            )

            abs_error = error.abs()

            mae = abs_error.mean()
            rmse = np.sqrt((error ** 2).mean())
            median_ae = abs_error.median()
            bias = error.mean()

        else:

            mae = np.nan
            rmse = np.nan
            median_ae = np.nan
            bias = np.nan

        rows.append(
            {
                group_col: g,
                "n_train": n_train,
                "events_train": n_event_train,
                "event_rate_train (%)": round(event_rate_train, 1),
                "n_test": n_test,
                "events_test": n_event_test,
                "event_rate_test (%)": round(event_rate_test, 1),
                "n_eval": n_eval,
                "MAE": round(mae, 2),
                "RMSE": round(rmse, 2),
                "MedianAE": round(median_ae, 2),
                "Bias": round(bias, 2),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
# ======================================================
# Construction d'un dataset d'analyse
# ======================================================

def build_analysis_dataset(dataset, predictions=None):
    """
    Prépare un dataset d'analyse (train ou test).

    Parameters
    ----------
    dataset : Polars DataFrame
        Dataset train ou test.

    predictions : pandas.DataFrame, optional
        Les prédictions XGBoost.
        Si None -> dataset train.
        Sinon -> dataset test avec fusion des prédictions.

    Returns
    -------
    pandas.DataFrame
    """

    # ----------------------------------------------------
    # Colonnes à récupérer
    # ----------------------------------------------------

    cols = [
        "patient_id",
        "genetic",
        "current_ckd_epi",
        "age_creat",
    ]

    # Pour le TRAIN uniquement, on a besoin de event_irt
    if predictions is None:
        cols.append("event_irt")

    info = (
        dataset
        .select(cols)
        .to_pandas()
    )

    # ----------------------------------------------------
    # Cas TRAIN
    # ----------------------------------------------------

    if predictions is None:

        df = info.copy()

    # ----------------------------------------------------
    # Cas TEST
    # ----------------------------------------------------

    else:

        # predictions_xgb contient déjà event_irt
        df = info.merge(
            predictions,
            on="patient_id",
            how="left"
        )

    # ----------------------------------------------------
    # Stade CKD
    # ----------------------------------------------------

    df["ckd_stage"] = np.select(

        [
            df["current_ckd_epi"] >= 90,
            (df["current_ckd_epi"] >= 60) & (df["current_ckd_epi"] < 90),
            (df["current_ckd_epi"] >= 30) & (df["current_ckd_epi"] < 60),
        ],

        [
            "CKD 1 (≥90)",
            "CKD 2 (60–89)",
            "CKD 3 (30–59)",
        ],

        default="CKD 4–5 (<30)"

    )

    # ----------------------------------------------------
    # Groupe d'âge
    # ----------------------------------------------------

    df["age_group"] = np.select(

        [
            df["age_creat"] < 40,
            (df["age_creat"] >= 40) & (df["age_creat"] < 50),
            (df["age_creat"] >= 50) & (df["age_creat"] < 60),
        ],

        [
            "<40 ans",
            "40–49 ans",
            "50–59 ans",
        ],

        default="≥60 ans"

    )

    # ----------------------------------------------------
    # Groupe de temps prédit (TEST uniquement)
    # ----------------------------------------------------

    if predictions is not None:

        df["predicted_time_group"] = np.select(

            [
                df["pred_time"] < 5,
                (df["pred_time"] >= 5) & (df["pred_time"] < 10),
                (df["pred_time"] >= 10) & (df["pred_time"] < 15),
            ],

            [
                "<5 ans",
                "5–10 ans",
                "10–15 ans",
            ],

            default="≥15 ans"

        )

    return df

In [ ]:
# ===========================================================
# Résumé des performances par sous-groupe
# ===========================================================

def subgroup_performance(train_df, test_df, group_col):

    # Cas particulier : temps prédit (n'existe que dans le test)
    if group_col == "predicted_time_group":
        groups = sorted(test_df[group_col].dropna().unique())
    else:
        groups = sorted(
            set(train_df[group_col].dropna().unique())
            | set(test_df[group_col].dropna().unique())
        )

    rows = []

    for group in groups:

        # ==================================================
        # TRAIN
        # ==================================================

        if group_col == "predicted_time_group":

            n_train = np.nan
            n_train_events = np.nan

        else:

            train_group = train_df[train_df[group_col] == group]

            n_train = len(train_group)
            n_train_events = int(train_group["event_irt"].sum())

        # ==================================================
        # TEST
        # ==================================================

        test_group = test_df[test_df[group_col] == group]

        n_test = len(test_group)
        n_test_events = int(test_group["event_irt"].sum())

        # ==================================================
        # Performances
        # ==================================================

        events = test_group[(test_group["event_irt"] == 1)
             & (test_group["pred_time"].notna())]

        if len(events):

            error = events["pred_time"] - events["time_observed"]

            mae = np.abs(error).mean()
            rmse = np.sqrt((error**2).mean())
            bias = error.mean()

        else:

            mae = np.nan
            rmse = np.nan
            bias = np.nan

        rows.append({

            group_col: group,

            "n_train": n_train,
            "n_train_events": n_train_events,

            "n_test": n_test,
            "n_test_events": n_test_events,

            "MAE": mae,
            "RMSE": rmse,
            "Bias": bias,

        })

    return (
        pd.DataFrame(rows)
        .sort_values(group_col)
        .reset_index(drop=True)
        .round(2)
    )

In [ ]:
# =======================================================
# Calcul automatique des 12 snapshots
# =======================================================

tables_genetic = {}
tables_ckd = {}
tables_age = {}
tables_predicted = {}

for k in range(1, 13):

    # -----------------------------
    # Train
    # -----------------------------

    train_analysis = build_analysis_dataset(
        train_datasets[k]
    )

    # -----------------------------
    # Test
    # -----------------------------

    test_analysis = build_analysis_dataset(
        internal_test_datasets[k],
        predictions_xgb[k]
    )

    # -----------------------------
    # Génétique
    # -----------------------------

    tables_genetic[f"Snapshot {k}"] = subgroup_performance(
        train_analysis,
        test_analysis,
        "genetic"
    )

    # -----------------------------
    # Stade CKD
    # -----------------------------

    tables_ckd[f"Snapshot {k}"] = subgroup_performance(
        train_analysis,
        test_analysis,
        "ckd_stage"
    )

    # -----------------------------
    # Âge
    # -----------------------------

    tables_age[f"Snapshot {k}"] = subgroup_performance(
        train_analysis,
        test_analysis,
        "age_group"
    )

    # -----------------------------
    # Temps prédit
    # -----------------------------

    tables_predicted[f"Snapshot {k}"] = subgroup_performance(
        train_analysis,
        test_analysis,
        "predicted_time_group"
    )

In [ ]:
# =====================================================
# Dictionnaire des tableaux
# =====================================================

tables = {
    "🧬 Génétique": tables_genetic,
    "🩺 Stade CKD": tables_ckd,
    "👤 Âge": tables_age,
    "⏳ Temps prédit": tables_predicted,
}

# =====================================================
# Onglets principaux
# =====================================================

main_tabs = widgets.Tab()

main_outputs = []

for subgroup_name, subgroup_tables in tables.items():

    # -------------------------------------------------
    # Sous-onglets (Snapshots)
    # -------------------------------------------------

    snapshot_tabs = widgets.Tab()

    snapshot_outputs = []

    for k in range(1, 13):

        out = widgets.Output()

        with out:
            display(subgroup_tables[f"Snapshot {k}"])

        snapshot_outputs.append(out)

    snapshot_tabs.children = snapshot_outputs

    for k in range(12):
        snapshot_tabs.set_title(k, f"Snapshot {k+1}")

    main_outputs.append(snapshot_tabs)

# =====================================================
# Assemblage
# =====================================================

main_tabs.children = main_outputs

for i, title in enumerate(tables.keys()):
    main_tabs.set_title(i, title)

display(main_tabs)

### Analyse des patients bien et mal prédits

In [ ]:
# ======================================================
# Meilleurs et pires patients par snapshot
# ======================================================

best_patients = {}
worst_patients = {}

cols = [

    # Identification
    "patient_id",

    # Profil du patient
    "genetic",
    "age_creat",
    "age_group",

    # État rénal
    "current_ckd_epi",
    "ckd_stage",

    # Temps réel
    "time_observed",

    # Prédiction
    "pred_time",
    "predicted_time_group",

    # Erreurs
    "error",
    "abs_error",

]

for k in range(1, 13):

    # ----------------------------
    # Dataset d'analyse
    # ----------------------------
    analysis = build_analysis_dataset(
        internal_test_datasets[k],
        predictions_xgb[k]
    )

    # Uniquement les événements prédits
    analysis = analysis[
        (analysis["event_irt"] == 1)
        & (analysis["pred_time"].notna())
    ].copy()

    # ----------------------------
    # Calcul des erreurs
    # ----------------------------
    analysis["error"] = (
        analysis["pred_time"] - analysis["time_observed"]
    )

    analysis["abs_error"] = analysis["error"].abs()

    # ----------------------------
    # 20 meilleurs patients
    # ----------------------------
    best_patients[f"Snapshot {k}"] = (
        analysis
        .sort_values("abs_error")
        .head(10)[cols]
        .reset_index(drop=True)
        .round(2)
    )

    # ----------------------------
    # 20 pires patients
    # ----------------------------
    worst_patients[f"Snapshot {k}"] = (
        analysis
        .sort_values("abs_error", ascending=False)
        .head(10)[cols]
        .reset_index(drop=True)
        .round(2)
    )

In [ ]:
tables = {
    "🏆 10 meilleurs": best_patients,
    "⚠️ 10 pires": worst_patients,
}

main_tabs = widgets.Tab()
main_outputs = []

for title, snapshot_tables in tables.items():

    snapshot_tabs = widgets.Tab()
    outputs = []

    for k in range(1, 13):

        out = widgets.Output()

        with out:
            display(snapshot_tables[f"Snapshot {k}"])

        outputs.append(out)

    snapshot_tabs.children = outputs

    for k in range(12):
        snapshot_tabs.set_title(k, f"Snapshot {k+1}")

    main_outputs.append(snapshot_tabs)

main_tabs.children = main_outputs

for i, title in enumerate(tables.keys()):
    main_tabs.set_title(i, title)

display(main_tabs)

### SHAP

In [ ]:

shap_explanations = {}

for k in range(1, 13):

    print(f"Snapshot {k}")

    X = X_train_xgb[k]
    booster = models_xgb[k]

    dmatrix = xgb.DMatrix(
        X,
        enable_categorical=True
    )

    contrib = booster.predict(
        dmatrix,
        pred_contribs=True
    )

    values = contrib[:, :-1]
    base_values = contrib[:, -1]

    shap_explanations[k] = shap.Explanation(
        values=values,
        base_values=base_values,
        data=X,
        feature_names=X.columns.tolist()
    )

print("✅ SHAP calculé pour les 12 snapshots.")


In [ ]:
out_beeswarm = widgets.Output()

def plot_beeswarm(snapshot):

    with out_beeswarm:

        clear_output(wait=True)

        shap.plots.beeswarm(
            shap_explanations[snapshot],
            max_display=20,
            show=True
        )

dropdown_beeswarm = widgets.Dropdown(
    options=[(f"Snapshot {k}", k) for k in range(1, 13)],
    value=1,
    description="Snapshot"
)

dropdown_beeswarm.observe(
    lambda change: plot_beeswarm(change["new"]),
    names="value"
)

display(dropdown_beeswarm)
display(out_beeswarm)

plot_beeswarm(1)

In [ ]:
# =================================================
# sauvegarde le beeswarm d'un snapshot précis
# =================================================

snapshot = 6

plt.figure(figsize=(10, 8))

shap.plots.beeswarm(
    shap_explanations[snapshot],
    max_display=20,
    show=False
)

plt.title("SHAP Beeswarm - Snapshot 6", fontsize=16, pad=15)

plt.tight_layout()

plt.savefig(
    f"beeswarm_snapshot_{snapshot}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
out_bar = widgets.Output()

def plot_bar(snapshot):

    with out_bar:

        clear_output(wait=True)

        explanation = shap_explanations[snapshot]

        importance = (
            pd.DataFrame({
                "Variable": explanation.feature_names,
                "Importance": np.abs(explanation.values).mean(axis=0)
            })
            .sort_values("Importance", ascending=True)
        )

        fig = px.bar(

            importance,

            x="Importance",

            y="Variable",

            orientation="h",

            color_discrete_sequence=["royalblue"],

            title=f"Importance des variables - Snapshot {snapshot}"

        )

        fig.update_layout(

            template="plotly_white",

            width=900,

            height=700,

            showlegend=False,

            xaxis_title="Mean(|SHAP|)",

            yaxis_title=""

        )

        fig.show()

dropdown_bar = widgets.Dropdown(

    options=[(f"Snapshot {k}", k) for k in range(1, 13)],

    value=1,

    description="Snapshot"

)

dropdown_bar.observe(

    lambda change: plot_bar(change["new"]),

    names="value"

)

display(dropdown_bar)
display(out_bar)

plot_bar(1)


In [ ]:
importance_table = pd.DataFrame()

for k in range(1, 13):

    expl = shap_explanations[k]

    importance_table[f"Snapshot {k}"] = pd.Series(
        np.abs(expl.values).mean(axis=0),
        index=expl.feature_names
    )

importance_table.index.name = "Variable"

importance_table



# tableau des ranges des variables par snapshot

rank_table = importance_table.rank(
    axis=0,
    ascending=False,
    method="min"
).astype(int)

# Trier les lignes selon le classement du snapshot 1
rank_table = rank_table.sort_values("Snapshot 1")

rank_table


fig = px.imshow(
    importance_table,
    color_continuous_scale="Blues",
    aspect="auto",
    labels={
        "x": "Snapshot",
        "y": "Variable",
        "color": "Mean(|SHAP|)"
    },
    title="Évolution de l'importance des variables au fil des snapshots"
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=700
)

fig.show()

In [ ]:
# ==========================================================
# Heatmap normalisée par variable
# ==========================================================

importance_norm = importance_table.copy()

# Normalisation ligne par ligne
importance_norm = importance_norm.div(
    importance_norm.max(axis=1).replace(0, 1),
    axis=0
)

fig = px.imshow(

    importance_norm,

    color_continuous_scale="Blues",

    aspect="auto",

    labels={
        "x": "Snapshot",
        "y": "Variable",
        "color": "Importance\nrelative"
    },

    title="Évolution relative de l'importance des variables au fil des snapshots"

)

fig.update_layout(

    template="plotly_white",

    width=1000,

    height=700

)

fig.show()

## VALIDATION EXTERNE XGBOOST SURVIVAL AFT

In [ ]:
# ==========================================================
# Fonction de validation externe
# ==========================================================

def run_xgb_survival_external(

    model,

    train_dataset,

    test_dataset,

    features,

    k,

    verbose=True

):

    # ------------------------------------------------------
    # Préparation des données
    # ------------------------------------------------------

    (

        X_train,

        X_test,

        y_train_df,

        y_test_df,

        test_info

    ) = prepare_dataset(

        train_dataset,

        test_dataset,

        features

    )

    # ------------------------------------------------------
    # Vérification des dimensions
    # ------------------------------------------------------

    assert len(X_train) == len(y_train_df)

    assert len(X_test) == len(y_test_df)

    # ------------------------------------------------------
    # Labels AFT
    # ------------------------------------------------------

    _, _ = prepare_aft_labels(
        y_train_df
    )

    test_lower, test_upper = prepare_aft_labels(
        y_test_df
    )

    # ------------------------------------------------------
    # DMatrix externe
    # ------------------------------------------------------

    dtest = build_dmatrix(

        X_test,

        test_lower,

        test_upper

    )

    # ------------------------------------------------------
    # Prédictions
    # ------------------------------------------------------

    pred_test = model.predict(
        dtest
    )

    # ------------------------------------------------------
    # Évaluation
    # ------------------------------------------------------

    evaluation = evaluate_predictions(

        test_info=test_info,

        pred_time=pred_test,

        k=k

    )

    # ------------------------------------------------------
    # Historique
    # ------------------------------------------------------

    history = (

        evaluation["predictions"]

        .copy()

        .assign(

            n_visits=k

        )

    )

    # ------------------------------------------------------
    # Affichage
    # ------------------------------------------------------

    if verbose:

        print("=" * 60)

        print("Validation externe - XGBoost Survival AFT")

        print("=" * 60)

        print(f"C-index : {evaluation['cindex']:.4f}")

        print(f"Patients : {evaluation['n_patients']}")

        print(f"Événements : {evaluation['n_events']}")

        print(f"Patients évalués : {evaluation['n_predicted']}")

        print(f"MAE : {evaluation['mae']:.2f} années")

        print(f"RMSE : {evaluation['rmse']:.2f} années")

    # ------------------------------------------------------
    # Résultats
    # ------------------------------------------------------

    return {

        "test_info": test_info,

        "history": history,

        "event_predictions": evaluation["predictions"],

        "cindex_test": evaluation["cindex"],

        "mae": evaluation["mae"],

        "rmse": evaluation["rmse"],

        "n_patients": evaluation["n_patients"],

        "n_events": evaluation["n_events"],

        "n_predicted": evaluation["n_predicted"]

    }

In [ ]:
# ==========================================================
# VALIDATION EXTERNE
# XGBOOST SURVIVAL AFT
# ==========================================================

results_xgb_external = []

predictions_xgb_external = {}

models_xgb_external = {}

X_train_xgb_external = {}

X_test_xgb_external = {}

test_info_xgb_external = {}

for k in range(1, 13):

    res = run_xgb_survival(

        train_dataset=train_datasets[k],

        test_dataset=external_test_datasets[k],

        features=FEATURES,

        k=k,

        verbose=False

    )

    models_xgb_external[k] = res["model"]

    X_train_xgb_external[k] = res["X_train"]

    X_test_xgb_external[k] = res["X_test"]

    test_info_xgb_external[k] = res["test_info"]

    predictions_xgb_external[k] = res["event_predictions"]

    results_xgb_external.append({

        "model": "XGB",

        "n_visits": k,

        "n_patients": res["n_patients"],

        "n_events": res["n_events"],

        "n_predicted": res["n_predicted"],

        "cindex_train": res["cindex_train"],

        "cindex_test": res["cindex_test"],

        "mae": res["mae"],

        "rmse": res["rmse"]

    })

results_xgb_external = pd.DataFrame(results_xgb_external)

display(results_xgb_external)

In [ ]:
# ==========================================================
# SAUVEGARDE HTML — XGBOOST SURVIVAL AFT
# VALIDATION INTERNE & EXTERNE
# ==========================================================

import os

# ----------------------------------------------------------
# Dossier de sortie
# ----------------------------------------------------------

output_dir = (
    r".\tables\survival_models"
)

os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------------------------
# Chemin du fichier HTML
# ----------------------------------------------------------

html_path = os.path.join(
    output_dir,
    "xgboost_survival_aft_longitudinal.html"
)

# ----------------------------------------------------------
# Génération du HTML
# ----------------------------------------------------------

html_content = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="utf-8">

<title>XGBoost Survival AFT Results</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 40px;
    background-color: white;
}}

h1 {{
    color: #222222;
}}

h2 {{
    margin-top: 50px;
    color: #333333;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 15px;
    margin-bottom: 40px;
}}

th,
td {{
    border: 1px solid #cccccc;
    padding: 8px;
    text-align: center;
}}

th {{
    background-color: #f2f2f2;
}}

tr:nth-child(even) {{
    background-color: #fafafa;
}}

</style>

</head>

<body>

<h1>XGBoost Survival AFT Results</h1>

<h2>Internal Validation</h2>

{results_xgb_internal.to_html(index=False)}

<h2>External Validation</h2>

{results_xgb_external.to_html(index=False)}

</body>

</html>
"""

# ----------------------------------------------------------
# Sauvegarde
# ----------------------------------------------------------

with open(
    html_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(html_content)

print(f"HTML sauvegardé :\n{html_path}")